# Seasonal observing plan

This is the end-to-end planning workflow. It loads candidates from configured TOMs, collects their available photometry (including matched Rubin DiaObject/DiaSource measurements when enabled), computes visibility, and calls the single evidence-based priority evaluator. The result is an auditable category/rank table for every candidate plus a separate season-feasible list. No weighted priority score is used.


In [1]:
from pathlib import Path
import pandas as pd
from target_selection.cache_paths import OUTPUT_ROOT, CACHE_ROOT, MOP_PHOTOMETRY_CACHE
from target_selection.planning import (load_candidates, add_ogle_event_parameters, collect_photometry, add_latest_magnitude, evaluate_visibility, visibility_plots, save_priority_products, save_target_sky_map, save_planning_pdf)
from target_selection.prioritization import evaluate_priority, PriorityConfig, SCHEDULABLE_CATEGORIES

START_DATE = '2026-10-16'
END_DATE = '2026-10-18'
OBSERVATORY = 'El Leoncito'
PROVIDERS = None  # None loads all registered TOMs; set a tuple to restrict the run.
MIN_ALTITUDE_DEG = 40.0
MIN_VISIBLE_MINUTES = 60.0
LOAD_PHOTOMETRY = True
LOAD_LSST_PHOTOMETRY = True  # Query DP2 DiaObject/DiaSource photometry before assessment.
MAKE_LIGHTCURVES = True
MAKE_LSST_CUTOUTS = False
PRIORITY_CONFIG = PriorityConfig(planned_visit_interval_days=7.0)
OUTPUT = OUTPUT_ROOT / 'notebooks' / 'seasonal_observing_plan' / f'{START_DATE}_to_{END_DATE}'
CACHE = CACHE_ROOT
OUTPUT.mkdir(parents=True, exist_ok=True)


Load and deduplicate the candidate events first. Fill missing `t0`, `tE`, and `u0` from the matching OGLE EWS event metadata when available. Photometry and visibility then use the full candidate set so events are not dropped before scientific assessment.


In [2]:
targets = load_candidates(PROVIDERS, start_date=START_DATE, end_date=END_DATE, include_photometry=LOAD_PHOTOMETRY, cache_dir=CACHE, output_dir=OUTPUT)
targets = add_ogle_event_parameters(targets)
print(f'Loaded targets: {len(targets):,}')
display(targets.head())

[planning] bhtom: 657 targets
[targets] Querying MOP visibility for 2026-10-16 to 2026-10-18.
[targets] MOP targets with event data: 37
[planning] mop: 37 targets
[planning] opm: 185 targets
[planning] OGLE event parameters unavailable: Expected OGLE-YYYY-FIELD-NNNN event ID, got '2026-BLG-0001'
Loaded targets: 877


,target_id,type,has_optical,has_infrared,has_radio,has_xray,has_gamma,has_polarimetry,created,modified,...,ANTARES microlensing filter,Peak Planet Probability from fit ψ Rescaled,Bogus ALeRCE stamp,opm_updated,opm_target_links,opm_target_page_id,opm_target_url,band_last,source_providers,input_sources
0,Gaia21fkl,SIDEREAL,True,True,False,False,True,False,2023-10-03T02:04:39.458Z,2026-04-07T17:28:29.973Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Gaia18ajz,SIDEREAL,True,True,False,False,True,False,2023-10-04T09:46:43.417Z,2026-02-22T16:53:32.822Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Gaia19dke,SIDEREAL,True,True,False,False,True,False,2023-10-04T09:53:49.044Z,2026-02-22T16:53:32.890Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Gaia23ats,SIDEREAL,True,True,False,False,True,False,2023-10-05T10:14:02.086Z,2026-02-22T16:53:33.022Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Gaia22daq,SIDEREAL,True,True,False,False,True,False,2023-10-05T10:15:27.456Z,2026-02-22T16:53:33.156Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Evaluate visibility

Visibility is an operational constraint, separate from scientific category. The nightly table records time available above the adopted altitude and duration limits.


In [3]:
visibility_by_night, visibility_summary, visible_targets = evaluate_visibility(targets, OUTPUT / 'visibility', start_date=START_DATE, end_date=END_DATE, observatory=OBSERVATORY, minimum_altitude_deg=MIN_ALTITUDE_DEG, minimum_observable_minutes=MIN_VISIBLE_MINUTES)
print(f'Visible targets before photometry: {len(visible_targets):,}/{len(targets):,}')
display(visible_targets.head())

[visibility] Evaluating 877 targets from 2026-10-16 to 2026-10-18.
[visibility] Rows: 2631; selected: 253.
Visible targets before photometry: 93/877


,target_id,type,has_optical,has_infrared,has_radio,has_xray,has_gamma,has_polarimetry,created,modified,...,ANTARES microlensing filter,Peak Planet Probability from fit ψ Rescaled,Bogus ALeRCE stamp,opm_updated,opm_target_links,opm_target_page_id,opm_target_url,band_last,source_providers,input_sources
0,Gaia21fkl,SIDEREAL,True,True,False,False,True,False,2023-10-03T02:04:39.458Z,2026-04-07T17:28:29.973Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11,Gaia23air,SIDEREAL,True,True,False,False,True,False,2023-10-12T08:20:06.505Z,2026-02-22T16:53:34.560Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12,Gaia23cnu,SIDEREAL,True,True,False,False,True,False,2023-10-12T11:32:16.281Z,2026-02-22T16:53:34.717Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,Gaia20fnr,SIDEREAL,True,True,False,False,True,False,2023-10-24T14:46:08.388Z,2026-02-22T16:56:39.003Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
34,Gaia23cbf,SIDEREAL,True,True,False,False,True,False,2023-10-25T17:24:19.789Z,2026-02-22T16:57:16.007Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Collect photometry from all candidate sources

Collect TOM and associated-survey photometry for the full candidate set. When enabled, query Rubin DP2 around every candidate coordinate and normalize its epochs and source labels before combining the rows.


In [4]:
observations = collect_photometry(
    targets, providers=PROVIDERS, include_associated_surveys=True,
    cache_dir=CACHE, output_dir=OUTPUT,
) if LOAD_PHOTOMETRY else pd.DataFrame()

lsst_photometry = pd.DataFrame()
if LOAD_LSST_PHOTOMETRY and not targets.empty:
    from target_selection import data_source
    lsst_input = targets.rename(columns={'target_id': 'Target', 'ra_deg': 'RA_deg', 'dec_deg': 'Dec_deg'}).copy()
    lsst_photometry = data_source.lsst.photometry(
        lsst_input, method='dia_combined_catalog',
        output_path=OUTPUT / 'lsst_dp2_photometry.csv', verbose=True,
    )
    if 'Target' in lsst_photometry and 'target_id' not in lsst_photometry:
        lsst_photometry = lsst_photometry.rename(columns={'Target': 'target_id'})
    if 'expMidptMJD' in lsst_photometry and 'epoch_mjd' not in lsst_photometry:
        lsst_photometry['epoch_mjd'] = pd.to_numeric(lsst_photometry['expMidptMJD'], errors='coerce')
    if 'Source' in lsst_photometry and 'survey' not in lsst_photometry:
        lsst_photometry['survey'] = lsst_photometry['Source']
    lsst_photometry['provider'] = 'lsst'
    lsst_photometry = lsst_photometry.loc[
        lsst_photometry.get('measurement_status', pd.Series('measured', index=lsst_photometry.index)).eq('measured')
    ].copy()
    observations = pd.concat([observations, lsst_photometry], ignore_index=True, sort=False)

if not observations.empty:
    targets = add_latest_magnitude(targets, observations, output_dir=OUTPUT)
print(f'Candidates: {len(targets):,}; photometry rows: {len(observations):,}; LSST rows: {len(lsst_photometry):,}')


[planning] bhtom: 1,553 photometry rows
[planning] mop: 304 photometry rows
[planning] opm: 43 photometry rows
[planning] ogle: 10,684 complementary photometry rows
[lsst] DiaSource rows returned: 162 (161 measured; 10 targets).
[planning] lsst: 4,731 DiaSource/forced photometry rows


/home/karennowo/target_selection/target_selection/sources/lsst.py:185: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(nonempty, ignore_index=True, sort=False) if nonempty else pd.DataFrame()


[lsst] DiaSource rows returned: 1840 (1118 measured; 877 targets).


/home/karennowo/target_selection/target_selection/sources/lsst.py:185: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(nonempty, ignore_index=True, sort=False) if nonempty else pd.DataFrame()


Candidates: 877; photometry rows: 29,751; LSST rows: 12,790


## 4. Assess evidence and assign categories

This is the sole priority method. It returns a category, within-category and global ranks, metric columns, and a reason for every candidate. Visibility and brightness limits remain separate feasibility evidence. `REVISAR` marks insufficient evidence instead of forcing a scientific classification.


In [5]:
priority = evaluate_priority(
    targets, observations, visibility_summary,
    as_of=START_DATE, config=PRIORITY_CONFIG,
    output_dir=OUTPUT / 'tables',
)
priority_csv, priority_png = save_priority_products(
    priority, OUTPUT / 'tables', name='priority_assessment', title='Evidence-based priority assessment',
    subtitle='Categories and ranks come from the shared rule-based evaluator; visibility and magnitude are feasibility evidence.',
    write_csv=False,
)

visible = pd.to_numeric(priority.get('n_visible_nights', pd.Series(0, index=priority.index)), errors='coerce').fillna(0).gt(0)
bright_enough = (
    priority.get('passes_hsh_magnitude_limit', pd.Series(False, index=priority.index)).fillna(False).astype(bool)
    | priority.get('passes_js_magnitude_limit', pd.Series(False, index=priority.index)).fillna(False).astype(bool)
)
candidates = priority.loc[priority['priority_category'].isin(SCHEDULABLE_CATEGORIES) & visible & bright_enough].copy()
candidates.to_csv(OUTPUT / 'nightly_schedule_candidates.csv', index=False)
sky_map = save_target_sky_map(candidates, OUTPUT / 'sky_map.png', title='Season candidates after priority assessment')
visibility_plot_paths = visibility_plots(
    candidates, OUTPUT / 'visibility' / 'plots', start_date=START_DATE,
    end_date=END_DATE, observatory=OBSERVATORY,
    minimum_altitude_deg=MIN_ALTITUDE_DEG,
    minimum_observable_minutes=MIN_VISIBLE_MINUTES,
    selection=visibility_by_night,
)
visibility_png = OUTPUT / 'visibility' / 'visibility_summary.png'
review_columns = ['priority_rank', 'category_rank', 'target_id', 'priority_category', 'priority_reason', 'event_stage', 'event_model_parameters_available', 't0_hjd', 'tE_days', 'last_external_survey', 'days_since_external_photometry', 'external_stale_threshold_days', 'stale_external_data', 'magnitude_assessment', 'n_visible_nights', 'best_observable_minutes']
display(priority[[column for column in review_columns if column in priority]].head(25))


      visibility: 1/3 (2026-10-16, 0/13 selected, 0 plot(s))
      visibility: 2/3 (2026-10-17, 0/13 selected, 0 plot(s))
      visibility: 3/3 (2026-10-18, 0/13 selected, 0 plot(s))


,priority_rank,category_rank,target_id,priority_category,priority_reason,event_stage,event_model_parameters_available,t0_hjd,tE_days,last_external_survey,days_since_external_photometry,external_stale_threshold_days,stale_external_data,magnitude_assessment,n_visible_nights,best_observable_minutes
0,1,1,OGLE-2024-BLG-0808,DIAGNÓSTICO,evento plausible con fotometría externa desact...,decline,True,2.460630e+06,408.70429,lsst,448.782942,30.0,True,16.492000,3.0,39.0
1,2,2,OGLE-2025-BLG-1121,DIAGNÓSTICO,evento plausible con fotometría externa desact...,decline,True,2.460873e+06,277.68400,ogle,367.000120,30.0,True,19.561000,3.0,2.0
2,3,3,OGLE-2025-BLG-0242,DIAGNÓSTICO,evento plausible con fotometría externa desact...,decline,True,2.460781e+06,670.72500,ogle,363.997810,30.0,True,17.148000,3.0,0.0
3,4,4,OGLE-2025-BLG-1217,DIAGNÓSTICO,evento plausible con fotometría externa desact...,decline,True,2.460964e+06,231.08828,ogle,363.996230,30.0,True,17.853000,3.0,2.0
4,5,5,OGLE-2025-BLG-0839,DIAGNÓSTICO,evento plausible con fotometría externa desact...,unknown,False,0.000000e+00,0.00000,ogle,363.994760,30.0,True,17.450000,3.0,0.0
5,6,6,OGLE-2025-BLG-1435,DIAGNÓSTICO,evento plausible con fotometría externa desact...,decline,True,2.460976e+06,433.72237,ogle,363.971790,30.0,True,17.604000,3.0,0.0
6,7,7,OGLE-2025-BLG-0674,DIAGNÓSTICO,evento plausible con fotometría externa desact...,decline,True,2.460870e+06,799.26618,ogle,360.937710,30.0,True,19.653000,3.0,51.0
7,8,8,OGLE-2025-BLG-0450,DIAGNÓSTICO,evento plausible con fotometría externa desact...,decline,True,2.460786e+06,395.01220,ogle,360.006880,30.0,True,18.454000,3.0,6.0
8,9,9,OGLE-2025-BLG-0271,DIAGNÓSTICO,evento plausible con fotometría externa desact...,unknown,False,0.000000e+00,0.00000,ogle,359.970420,30.0,True,16.988000,3.0,45.0
9,10,10,OGLE-2026-GD-0001,DIAGNÓSTICO,evento plausible con fotometría externa desact...,decline,True,2.461142e+06,176.68659,ogle,132.932400,30.0,True,16.793000,3.0,0.0


## 5. Optional light curves and LSST image products

Light curves show the photometry used for classification, with survey/provider provenance retained. Cutouts are optional and requested only for the season-feasible candidate list.


In [6]:
lightcurve_pngs = []
if MAKE_LIGHTCURVES and not candidates.empty:
    from target_selection import report
    lightcurve_pngs = report.create_lightcurve_plots(
        candidates, output_dir=OUTPUT / 'lightcurves',
        mop_photometry_dir=MOP_PHOTOMETRY_CACHE,
        observatory_photometry=observations, release_photometry=lsst_photometry,
        layers=('mop_photometry', 'opm_photometry', 'ogle_photometry', 'bhtom_photometry', 'release_photometry'),
        observation_date=START_DATE, overwrite=True,
    )
print('Light-curve PNGs:', len(lightcurve_pngs))

lsst_cutout_manifest = None
if MAKE_LSST_CUTOUTS and not candidates.empty:
    from target_selection import data_source
    lsst_coverage = data_source.lsst.coverage(candidates, image_kinds=('individual', 'coadd'), properties=('maglim', 'psf_size', 'exposure_time'), output_path=OUTPUT / 'lsst_coverage.csv')
    lsst_cutout_manifest = data_source.lsst.get_image(candidates.merge(lsst_coverage, on='target_id', how='left'), kind='coadd', output_dir=OUTPUT / 'cutouts' / 'png', fits_dir=OUTPUT / 'cutouts' / 'fits', bands=tuple('ugrizy'), add_rgb=False)


ModuleNotFoundError: No module named 'monitoring_report'

## 6. Assemble the PDF

The PDF summarizes the assessment, visibility, and available light curves.


In [ ]:
cutout_images = sorted((OUTPUT / 'cutouts' / 'png').glob('*.png')) if MAKE_LSST_CUTOUTS else []
pdf_path = save_planning_pdf(
    OUTPUT / 'seasonal_observing_plan.pdf', sky_map=sky_map,
    table_images=(priority_png, visibility_png), visibility_images=visibility_plot_paths,
    lightcurve_images=lightcurve_pngs, cutout_images=cutout_images,
)
print(pdf_path)
